In [1]:
import h5py
import numpy as np
import pandas as pd
from astropy.io import fits
from tqdm import tqdm
import os
import healpy as hp
from ligo.skymap.io.fits import read_sky_map
from ligo.skymap.moc import uniq2nest, uniq2pixarea
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import torch

/fred/oz016/bgao_kn/rubin/lib/python3.10/site-packages/ligo/lw/lsctables.py:89: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal
/fred/oz016/bgao_kn/rubin/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# gw_df = pd.read_csv("/fred/oz016/bgao_kn/ML+GW+KN/dataset/O5_sim_bns/injections_final.csv")
SIM_NAME = "LSST_KN_BNS"
SIM_PATH = f"/fred/oz016/bgao_kn/SNANA/SNDATA_ROOT/SIM/{SIM_NAME}"
output_path = f"/fred/oz016/bgao_kn/data/{SIM_NAME}/"
BAND_MAP = {                   # Mapping filters to channel indices
    'LSST-u': 0, 'LSST-g': 1, 'LSST-r': 2, 'LSST-i': 3, 'LSST-z': 4, 'LSST-Y': 5,
}
NUM_BANDS = 6
MAX_LC_LENGTH = 200  # Maximum length of light curves all band

In [ ]:
gw_df = pd.read_csv("/fred/oz016/bgao_kn/ML+GW+KN/dataset/O5_sim_bns/injections_final.csv")
# process gw data
valid_index = []

for gw_idx, row in tqdm(gw_df.iterrows(), total=len(gw_df)):
    event_id = int(row['simulation_id'])
    sim_dir = os.path.join(SIM_PATH, f"{SIM_NAME}_{event_id}")
    try:
        with fits.open(os.path.join(sim_dir, f"{SIM_NAME}_{event_id}_HEAD.FITS")) as hdul:
            head_data = hdul[1].data
            if len(head_data) == 0:
                continue
    except Exception as e:  # simulation failed, path do not exists
        continue
    valid_index.append(gw_idx)

print(f"Total valid GW+Optical samples: {len(valid_index)}")
valid_gw_df = gw_df.loc[valid_index]
valid_gw_df.reset_index(drop=True, inplace=True)
valid_gw_df.head()

100%|██████████| 1822/1822 [01:30<00:00, 20.22it/s]

Total valid GW+Optical samples: 491


,simulation_id,mjd_time,gps_time,longitude,latitude,ra,dec,inclination,distance,mass1,...,radius2,compactness1,compactness2,mej_dyn,mej_wind,mej_total,phi,costheta,distmean,diststd
0,16,63197.6935,1.637599e+09,5.470332,0.136340,313.426919,7.811696,2.143329,381.31009,1.485862,...,11.917976,0.18476,0.17485,0.002886,0.023251,0.026138,37.288056,0.541763,646.749849,225.624429
1,22,61668.5650,1.505482e+09,0.373078,-0.435035,21.375821,-24.925671,2.883460,216.89843,1.356997,...,11.982318,0.16778,0.15428,0.002305,0.063343,0.065648,42.377880,0.966868,196.172658,40.278797
2,26,62098.6401,1.542641e+09,3.974584,0.496561,227.726883,28.450847,0.229814,287.12256,1.461213,...,11.877148,0.18148,0.18457,0.002955,0.020643,0.023598,32.458937,0.973709,471.566567,215.924137
3,42,62285.2002,1.558760e+09,2.905688,1.270784,166.483653,72.810571,1.970406,337.09581,1.490620,...,11.907870,0.18540,0.17762,0.002928,0.055248,0.058176,60.035095,0.389059,620.953390,264.967351
4,47,63324.4883,1.648554e+09,2.971165,-0.854202,170.235220,-48.942144,0.472468,531.54896,1.483816,...,11.799853,0.18448,0.19887,0.003441,0.031049,0.034489,38.094391,0.890448,515.547466,172.749053


In [3]:
valid_gw_df = pd.read_csv("/fred/oz016/bgao_kn/data/LSST_KN_BNS/gw_catalog.csv")
valid_gw_df

,simulation_id,mjd_time,gps_time,longitude,latitude,ra,dec,inclination,distance,mass1,...,compactness2,mej_dyn,mej_wind,mej_total,phi,costheta,distmean,diststd,NLIBID,NDET_LC
0,16,63197.6935,1.637599e+09,5.470332,0.136340,313.426919,7.811696,2.143329,381.31009,1.485862,...,0.17485,0.002886,0.023251,0.026138,37.288056,0.541763,646.749849,225.624429,3546,90
1,22,61668.5650,1.505482e+09,0.373078,-0.435035,21.375821,-24.925671,2.883460,216.89843,1.356997,...,0.15428,0.002305,0.063343,0.065648,42.377880,0.966868,196.172658,40.278797,7118,5098
2,26,62098.6401,1.542641e+09,3.974584,0.496561,227.726883,28.450847,0.229814,287.12256,1.461213,...,0.18457,0.002955,0.020643,0.023598,32.458937,0.973709,471.566567,215.924137,3725,18
3,42,62285.2002,1.558760e+09,2.905688,1.270784,166.483653,72.810571,1.970406,337.09581,1.490620,...,0.17762,0.002928,0.055248,0.058176,60.035095,0.389059,620.953390,264.967351,9690,57
4,47,63324.4883,1.648554e+09,2.971165,-0.854202,170.235220,-48.942144,0.472468,531.54896,1.483816,...,0.19887,0.003441,0.031049,0.034489,38.094391,0.890448,515.547466,172.749053,11031,881
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486,2289,62797.7510,1.603044e+09,2.880703,0.802580,165.052107,45.984446,2.183413,663.83052,1.608796,...,0.19337,0.003614,0.020183,0.023797,25.152672,0.575011,794.956390,250.356261,3917,2
487,2291,61744.4267,1.512037e+09,0.513252,-1.277873,29.407177,-73.216724,2.154036,151.96069,1.171929,...,0.17362,0.002701,0.025851,0.028552,70.867308,0.550731,217.157042,55.343842,9951,992
488,2297,61558.0001,1.495930e+09,1.866434,-0.526386,106.938814,-30.159677,1.046183,418.41066,1.340292,...,0.20041,0.003828,0.038988,0.042815,45.154976,0.500878,722.005345,253.606728,13040,2
489,2299,62771.2013,1.600750e+09,0.638760,0.464589,36.598236,26.619011,2.901249,828.31816,1.572015,...,0.19499,0.003490,0.032460,0.035951,55.177386,0.971256,897.907255,280.228321,6617,15


In [4]:
# find max_lc_length
max_lc_length = 0
max_id = -1
for gw_idx, row in tqdm(valid_gw_df.iterrows(), total=len(valid_gw_df)):
    event_id = int(row['simulation_id'])
    sim_dir = os.path.join(SIM_PATH, f"{SIM_NAME}_{event_id}")
    with fits.open(os.path.join(sim_dir, f"{SIM_NAME}_{event_id}_HEAD.FITS")) as hdul:
        head_data = hdul[1].data
        if len(head_data) == 0:
            continue
        if max(head_data['NOBS']) > max_lc_length:
            max_lc_length = max(head_data['NOBS'])
            max_id = event_id
print(f"Max LC length across all bands: {max_lc_length} for event {max_id}")

100%|██████████| 491/491 [00:20<00:00, 23.56it/s]

Max LC length across all bands: 216 for event 2222


In [5]:
# Functions for parsing SNANA FITS files, and sampling MOC skymaps
def parse_snana_fits(event_id, sim_dir, sim_name="LSST_KN_BNS"):
    """
    Parses {event_id}_HEAD.fits and {event_id}_PHOT.fits.
    Extracts multiple light curve realizations for a single GW event.
    
    Args:
        event_id: String ID of the event.
        sim_dir: Directory containing FITS files.
        
    Returns:
        List of tuples: [(values, masks, times), ...]
        Returns empty list if files are missing.
    """
    if type(event_id) == str:
        event_id = int(float(event_id))
    head_path = os.path.join(sim_dir, f"{sim_name}_{event_id}",f"{sim_name}_{event_id}_HEAD.FITS")
    phot_path = os.path.join(sim_dir, f"{sim_name}_{event_id}",f"{sim_name}_{event_id}_PHOT.FITS")

    if not os.path.exists(head_path) or not os.path.exists(phot_path):
        print(f"Warning: FITS files not found for {event_id}")
        return []

    try:
        # open readme file and get MJD explode value
        with open(os.path.join(sim_dir, f"{sim_name}_{event_id}",f"{sim_name}_{event_id}.README")) as f:
            readme_lines = f.readlines()
            mjd_explode = readme_lines[27].split(":")[1].split()[0]
            mjd_explode = float(mjd_explode)
        # Open FITS files
        with fits.open(head_path) as hdul_head, fits.open(phot_path) as hdul_phot:
            # Usually data is in extension 1
            data_head = hdul_head[1].data
            data_phot = hdul_phot[1].data
            
            # Use columns directly (Astropy FITS columns are case-insensitive usually)
            # HEAD columns
            ptrobs_min = data_head['PTROBS_MIN']
            ptrobs_max = data_head['PTROBS_MAX']
            
            # PHOT columns
            mjd_all = data_phot['MJD']
            flux_all = data_phot['FLUXCAL']
            fluxerr_all = data_phot['FLUXCALERR'] # Optional usage
            flt_all = data_phot['BAND'] # Filters
            
            extracted_lcs = []
            
            # Iterate over each realization in HEAD
            for i in range(len(data_head)):
                # SNANA uses 1-based indexing for pointers, Python uses 0-based
                # Start index: value - 1
                # End index: value (exclusive in python slicing)
                start_idx = ptrobs_min[i] - 1
                end_idx = ptrobs_max[i]
                
                # Slicing the PHOT data
                lc_mjd = mjd_all[start_idx : end_idx]
                lc_flux = flux_all[start_idx : end_idx]
                lc_fluxerr = fluxerr_all[start_idx : end_idx]
                lc_flt = flt_all[start_idx : end_idx]
                
                # --- Format Conversion (to Tensor-ready numpy) ---
                val_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)    # Values matrix (flux)
                err_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)    # Errors matrix (flux errors)
                mask_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)
                time_vec = np.zeros((MAX_LC_LENGTH,), dtype=np.float32)
                
                # 1. Time Normalization (Relative to BNS merger time)
                if len(lc_mjd) > 0:
                    rel_times = lc_mjd - mjd_explode
                else:
                    continue # Skip empty light curves

                # 2. Fill Matrices
                # Truncate if longer than MAX_LC_LENGTH
                seq_len = min(len(lc_mjd), MAX_LC_LENGTH)
                if len(lc_mjd) > MAX_LC_LENGTH:
                    print(f"Warning: Light curve for event {event_id} exceeds MAX_LC_LENGTH. Truncating.")
                    # Keep the MAX_LC_LENGTH points with smallest absolute rel_times
                    sorted_indices = np.argsort(np.abs(rel_times))[:MAX_LC_LENGTH]
                    sorted_indices = np.sort(sorted_indices)  # Sort back to chronological order
                    lc_mjd = lc_mjd[sorted_indices]
                    lc_flux = lc_flux[sorted_indices]
                    lc_fluxerr = lc_fluxerr[sorted_indices]
                    lc_flt = lc_flt[sorted_indices]
                    rel_times = rel_times[sorted_indices]
                
                for t in range(seq_len):
                    band_char = lc_flt[t].strip() # Remove whitespace
                    if band_char in BAND_MAP:
                        b_idx = BAND_MAP[band_char]
                        
                        val_mat[t, b_idx] = lc_flux[t]
                        err_mat[t, b_idx] = lc_fluxerr[t]
                        mask_mat[t, b_idx] = 1.0
                        time_vec[t] = rel_times[t]
                
                extracted_lcs.append((val_mat, err_mat, mask_mat, time_vec))
                
            return extracted_lcs

    except Exception as e:
        print(f"Error processing FITS for {event_id}: {e}")
        return []

def sample_moc_skymap(map_file):
    """
    Convert UNIQ to sky coordinates and calculte pixel areas.
    Note: Do not include DISTNORM in the output. For inf values in DISTMU, relace with distmean and diststd from metadata.
    """
    
    # read moc skymap and metadata
    moc_map = read_sky_map(map_file, moc=True, distances=True)
    _, meta = read_sky_map(map_file, nest=True)

    # extract distance meta info
    dist_mean = meta.get('distmean', None)
    dist_std = meta.get('diststd', None)

    uniq = moc_map['UNIQ']
    probdensity = moc_map['PROBDENSITY']
    distmu = moc_map['DISTMU']
    distsigma = moc_map['DISTSIGMA']
    distnorm = moc_map['DISTNORM']

    # 1) UNIQ -> order, ipix, nside
    order, ipix = uniq2nest(uniq)

    # 2) caculate pixel area
    dA = uniq2pixarea(uniq)
    # 3) calculate pixel probability
    dP = probdensity * dA
    # 4) calculate theta, phi
    ras  = np.zeros_like(ipix, dtype=np.float32)
    decs = np.zeros_like(ipix, dtype=np.float32)
    for k in np.unique(order):
        m = (order == k)
        this_ipix  = ipix[m]
        this_nside = 2 ** k
        theta, phi = hp.pix2ang(this_nside, this_ipix, nest=True)
        ras[m]  = np.degrees(phi)
        decs[m] = 90.0 - np.degrees(theta)
    
    # 5) return torch tensors
    gw_mocmap = torch.tensor(np.vstack([ras, decs,dA, dP, distmu, distsigma]), dtype=torch.float32)   # [6, N_pixels], no distnorm

    # 6) process unnormal distance values
    inf_dist_mu = torch.where(torch.isinf(gw_mocmap[4]))[0]
    gw_mocmap[4, inf_dist_mu] = dist_mean  # set inf to mean value
    gw_mocmap[5, inf_dist_mu] = dist_std   # set inf to std value

    return gw_mocmap  # [6, N_pixels]

In [7]:
lcs = parse_snana_fits(event_id=16, sim_dir="/fred/oz016/bgao_kn/SNANA/SNDATA_ROOT/SIM/LSST_KN_BNS")
lcs[0][1]

array([[ 0.      ,  0.      ,  0.      ,  0.      ,  0.      , 16.616879],
       [ 0.      ,  0.      ,  0.      ,  0.      , 17.19096 ,  0.      ],
       [ 0.      ,  0.      ,  0.      ,  0.      ,  0.      , 42.439476],
       ...,
       [ 0.      ,  0.      ,  0.      ,  0.      ,  0.      ,  0.      ],
       [ 0.      ,  0.      ,  0.      ,  0.      ,  0.      ,  0.      ],
       [ 0.      ,  0.      ,  0.      ,  0.      ,  0.      ,  0.      ]],
      dtype=float32)

In [12]:
# Function to create relational dataset
def create_relational_dataset(
    gw_catalog_path, 
    fits_dir, 
    output_h5_path
):
    """
    Main function to process all data and save to HDF5.
    """
    # 1. Load GW Catalog
    print(f"Loading GW Catalog from {gw_catalog_path}...")
    # Assuming CSV has columns: event_id, m1, m2, ..., skymap_path
    gw_df = pd.read_csv(gw_catalog_path)
    
    n_unique = len(gw_df)
    
    # 2. Initialize HDF5 File
    with h5py.File(output_h5_path, 'w') as f:
        # --- Group A: Unique GW Events ---
        grp_gw = f.create_group('events/gw_data')
        
        # Pre-allocate GW datasets (we know exact size N_unique)
        ds_gw_scalars = grp_gw.create_dataset('scalars', (n_unique, 9), dtype='f4')
        ds_gw_skymaps = grp_gw.create_dataset('skymaps', (n_unique, 6, 19200), dtype='f4') # 6 channels after cleaning
        # Store IDs as fixed-length ASCII strings
        dt_str = h5py.special_dtype(vlen=str) 
        ds_gw_ids = grp_gw.create_dataset('ids', (n_unique,), dtype=dt_str)
        
        # --- Group B: All Optical Data ---
        # We don't know total optical count yet, so we use resizable datasets (chunked)
        grp_opt = f.create_group('events/optical_data')
        
        chunk_size = 1024
        ds_opt_vals = grp_opt.create_dataset('values', (0, MAX_LC_LENGTH, NUM_BANDS), 
                                             maxshape=(None, MAX_LC_LENGTH, NUM_BANDS), dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS))
        ds_opt_errs = grp_opt.create_dataset('errors', (0, MAX_LC_LENGTH, NUM_BANDS),
                                             maxshape=(None, MAX_LC_LENGTH, NUM_BANDS), dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS))
        ds_opt_masks = grp_opt.create_dataset('masks', (0, MAX_LC_LENGTH, NUM_BANDS), 
                                              maxshape=(None, MAX_LC_LENGTH, NUM_BANDS), dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH, NUM_BANDS))
        ds_opt_times = grp_opt.create_dataset('times', (0, MAX_LC_LENGTH), 
                                              maxshape=(None, MAX_LC_LENGTH), dtype='f4', chunks=(chunk_size, MAX_LC_LENGTH))
        
        # Parent Index Mapping (The Relation)
        ds_parent_idx = grp_opt.create_dataset('parent_gw_idx', (0,), maxshape=(None,), dtype='i4', chunks=(chunk_size,))
        
        # --- Processing Loop ---
        print("Starting processing loop...")
        total_optical_count = 0
        
        # Buffer for optical data to reduce HDF5 resize calls (optimization)
        opt_buffer_vals = []
        opt_buffer_errs = []
        opt_buffer_masks = []
        opt_buffer_times = []
        opt_buffer_p_idx = []
        BUFFER_LIMIT = 5000 

        def flush_buffer():
            nonlocal total_optical_count, opt_buffer_vals, opt_buffer_errs, opt_buffer_masks, opt_buffer_times, opt_buffer_p_idx
            if len(opt_buffer_vals) == 0: return
            
            n_new = len(opt_buffer_vals)
            current_size = total_optical_count
            new_size = current_size + n_new
            
            # Resize datasets
            ds_opt_vals.resize(new_size, axis=0)
            ds_opt_errs.resize(new_size, axis=0)
            ds_opt_masks.resize(new_size, axis=0)
            ds_opt_times.resize(new_size, axis=0)
            ds_parent_idx.resize(new_size, axis=0)
            
            # Write data
            ds_opt_vals[current_size:new_size] = np.array(opt_buffer_vals)
            ds_opt_errs[current_size:new_size] = np.array(opt_buffer_errs)
            ds_opt_masks[current_size:new_size] = np.array(opt_buffer_masks)
            ds_opt_times[current_size:new_size] = np.array(opt_buffer_times)
            ds_parent_idx[current_size:new_size] = np.array(opt_buffer_p_idx)
            
            total_optical_count += n_new
            
            # Clear buffer
            opt_buffer_vals = []
            opt_buffer_errs = []
            opt_buffer_masks = []
            opt_buffer_times = []
            opt_buffer_p_idx = []

        # Iterate over unique GW events
        for gw_idx, row in tqdm(gw_df.iterrows(), total=n_unique):
            # if gw_idx > 1:
            #     break
            event_id = int(row['simulation_id'])
            # print(f"Processing GW Event {event_id} ({gw_idx+1}/{n_unique})...")
            
            # 1. Process & Save GW Data
            # Scalars (Columns m1...param14)
            # Adjust columns based on your CSV
            gw_params_name = ['mjd_time', 'gps_time', 'mass1', 'mass2', 'spin1z', 'spin2z', 'inclination', 'distmean', 'diststd']
            scalars = row[gw_params_name].values.astype(np.float32)
            ds_gw_scalars[gw_idx] = scalars
            ds_gw_ids[gw_idx] = str(event_id)
            
            # Skymap
            # Apply robust preprocessing (Returns Tensor [6, 19200])
            gw_mocmap = sample_moc_skymap(f"/fred/oz016/bgao_kn/data/bns_skymap/{event_id}.fits")
            ds_gw_skymaps[gw_idx] = gw_mocmap.numpy() # Convert back to numpy for HDF5
            
            # 2. Process Optical Data
            # Extract light curves from SNANA FITS
            lcs = parse_snana_fits(event_id, sim_dir=fits_dir)
            
            # Add to buffer
            for (vals, errs, masks, times) in lcs:
                opt_buffer_vals.append(vals)
                opt_buffer_errs.append(errs)
                opt_buffer_masks.append(masks)
                opt_buffer_times.append(times)
                opt_buffer_p_idx.append(gw_idx) # Link to parent GW index
            
            # Flush if buffer is full
            if len(opt_buffer_vals) >= BUFFER_LIMIT:
                flush_buffer()
        
        # Final flush
        flush_buffer()
        
        # Save metadata
        f.attrs['n_unique_gw'] = n_unique
        f.attrs['n_total_optical'] = total_optical_count
        print(f"\nProcessing Complete.")
        print(f"Unique GW Events: {n_unique}")
        print(f"Total Light Curves: {total_optical_count}")
        print(f"Saved to: {output_h5_path}")

In [13]:
create_relational_dataset(
    gw_catalog_path="/fred/oz016/bgao_kn/data/LSST_KN_BNS/gw_catalog.csv",
    fits_dir="/fred/oz016/bgao_kn/SNANA/SNDATA_ROOT/SIM/LSST_KN_BNS",
    output_h5_path="/fred/oz016/bgao_kn/data/LSST_KN_BNS/combined_dataset.h5"
)

Loading GW Catalog from /fred/oz016/bgao_kn/data/LSST_KN_BNS/gw_catalog.csv...
Starting processing loop...


  0%|          | 0/491 [00:00<?, ?it/s]

 32%|███▏      | 159/491 [00:47<01:02,  5.34it/s]

 85%|████████▌ | 418/491 [01:59<00:16,  4.48it/s]

 97%|█████████▋| 476/491 [02:30<00:05,  2.73it/s]

100%|██████████| 491/491 [02:36<00:00,  3.14it/s]



Processing Complete.
Unique GW Events: 491
Total Light Curves: 483659
Saved to: /fred/oz016/bgao_kn/data/LSST_KN_BNS/combined_dataset.h5


In [20]:
from collections import defaultdict
from typing import List, Iterator
from torch.utils.data import Dataset, DataLoader, Sampler

In [17]:
def build_gw_to_lc_mapping(h5_path: str):
    """
    Scans the HDF5 file to build a mapping from GW Event Index to Light Curve Indices.
    This is required for the Balanced Sampler.
    
    Args:
        h5_path: Path to the HDF5 file.
        
    Returns:
        gw_to_lc_map: Dictionary {gw_idx: np.array([lc_idx_1, lc_idx_2, ...])}
    """
    print(f"Building GW-to-Optical index mapping from {h5_path}...")
    with h5py.File(h5_path, 'r') as f:
        # Load the parent_gw_idx array into memory (it's essentially a list of integers)
        # Shape: [Total_Optical_Samples]
        all_parent_indices = f['events/optical_data/parent_gw_idx'][:]
        
    gw_to_lc_map = defaultdict(list)
    for lc_idx, gw_idx in enumerate(all_parent_indices):
        gw_to_lc_map[gw_idx].append(lc_idx)
        
    # Convert lists to numpy arrays for faster random sampling later
    final_map = {k: np.array(v) for k, v in gw_to_lc_map.items()}
    
    print(f"Mapping complete. Found {len(final_map)} unique GW events.")
    return final_map

In [19]:
gw_to_lc_map = build_gw_to_lc_mapping("/fred/oz016/bgao_kn/data/LSST_KN_BNS/combined_dataset.h5")
gw_to_lc_map[0]

Building GW-to-Optical index mapping from /fred/oz016/bgao_kn/data/LSST_KN_BNS/combined_dataset.h5...
Mapping complete. Found 491 unique GW events.


array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89])

In [ ]:
class RelationalHDF5Dataset(Dataset):
    """
    PyTorch Dataset for the relational HDF5 structure.
    Reads optical data by index and fetches the corresponding unique GW data.
    """
    def __init__(self, h5_path: str):
        super().__init__()
        self.h5_path = h5_path
        self.h5_file = None
        
        # Open file temporarily to get dataset length
        with h5py.File(h5_path, 'r') as f:
            self.length = f['events/optical_data/values'].shape[0]
            
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        """
        Args:
            idx: Index of the light curve (optical data).
        """
        # Lazy loading: Open file only when needed (crucial for num_workers > 0)
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
            
        # 1. Retrieve Optical Data (Values, Errors, Masks, Times)
        #    HDF5 structure: events/optical_data/...
        opt_val   = torch.from_numpy(self.h5_file['events/optical_data/values'][idx])
        opt_err   = torch.from_numpy(self.h5_file['events/optical_data/errors'][idx])
        opt_mask  = torch.from_numpy(self.h5_file['events/optical_data/masks'][idx])
        opt_time  = torch.from_numpy(self.h5_file['events/optical_data/times'][idx])
        
        # 2. Retrieve Parent GW Index
        gw_idx = self.h5_file['events/optical_data/parent_gw_idx'][idx]
        
        # 3. Retrieve Unique GW Data using gw_idx
        #    HDF5 structure: events/gw/...
        gw_scalar = torch.from_numpy(self.h5_file['events/gw_data/scalars'][gw_idx])
        gw_skymap = torch.from_numpy(self.h5_file['events/gw_data/skymaps'][gw_idx])
        
        # Return tuple: (GW_Inputs, Optical_Inputs, Metadata)
        # gw_idx is returned for masking the contrastive loss (handling same-source negatives)
        return gw_scalar, gw_skymap, opt_time, opt_val, opt_mask, opt_err, int(gw_idx)
    
class BalancedGWBatchedSampler(Sampler):
    """
    Custom Batch Sampler that ensures:
    1. Each batch contains 'batch_size' UNIQUE GW events.
    2. For each selected GW event, ONE light curve is randomly sampled.
    
    This prevents "false negatives" in contrastive learning where multiple LCs 
    from the same GW event appear in the same batch.
    """
    def __init__(self, gw_to_lc_map: dict, batch_size: int, steps_per_epoch: int):
        """
        Args:
            gw_to_lc_map: Dictionary mapping GW_ID -> [LC_ID_1, LC_ID_2, ...]
            batch_size: Number of unique GW events per batch.
            steps_per_epoch: Number of batches to yield per 'epoch'. (N_LC // batch_size)
        """
        self.gw_to_lc_map = gw_to_lc_map
        self.unique_gw_ids = list(gw_to_lc_map.keys())
        self.batch_size = batch_size
        self.steps_per_epoch = steps_per_epoch
        
        # Validation: Batch size cannot exceed total unique GW events
        if self.batch_size > len(self.unique_gw_ids):
            raise ValueError(f"Batch size ({batch_size}) > Unique GW events ({len(self.unique_gw_ids)}).")

    def __iter__(self) -> Iterator[List[int]]:
        for _ in range(self.steps_per_epoch):
            # 1. Sample unique GW IDs for this batch (without replacement)
            batch_gw_ids = np.random.choice(
                self.unique_gw_ids, 
                size=self.batch_size, 
                replace=False
            )
            
            batch_lc_indices = []
            
            # 2. For each GW ID, sample ONE random light curve index
            for gw_id in batch_gw_ids:
                possible_lcs = self.gw_to_lc_map[gw_id]
                chosen_lc = np.random.choice(possible_lcs)
                batch_lc_indices.append(chosen_lc)
                
            # Yield the list of optical indices for the DataLoader to fetch
            yield batch_lc_indices

    def __len__(self):
        return self.steps_per_epoch

def create_training_dataloader(
    h5_path: str, 
    batch_size: int = 32, 
    steps_per_epoch: int = 1000, 
    num_workers: int = 4
):
    """
    Factory function to initialize the Dataset, Sampler, and DataLoader.
    """
    # 1. Build Index Map (Once)
    gw_map = build_gw_to_lc_mapping(h5_path)
    
    # 2. Initialize Dataset
    dataset = RelationalHDF5Dataset(h5_path)
    
    # 3. Initialize Custom Sampler
    # Note: 'steps_per_epoch' defines how many batches constitute one epoch loop
    sampler = BalancedGWBatchedSampler(
        gw_to_lc_map=gw_map,
        batch_size=batch_size,
        steps_per_epoch=steps_per_epoch
    )
    
    # 4. Initialize DataLoader
    # IMPORTANT: batch_sampler is used, so batch_size/shuffle/sampler/drop_last 
    # arguments in DataLoader constructor must not be provided.
    loader = DataLoader(
        dataset,
        batch_sampler=sampler,
        num_workers=num_workers,
        pin_memory=True
    )
    
    return loader

In [29]:
h5file = "/fred/oz016/bgao_kn/data/LSST_KN_BNS/combined_dataset.h5"
dataloader = create_training_dataloader(
    h5_path=h5file,
    batch_size=32,
    steps_per_epoch=10000
)
print("\nTesting DataLoader...")
for batch_idx, batch_data in enumerate(dataloader):
    # Unpack data
    # Order matches __getitem__: scalar, skymap, time, val, mask, err, gw_idx
    gw_s, gw_m, opt_t, opt_v, opt_mask, opt_err, gw_indices = batch_data
    
    print(f"Batch {batch_idx}:")
    print(f"  GW Scalars: {gw_s.shape}")    # Expected: [B, 9]
    print(f"  GW Skymap:  {gw_m.shape}")    # Expected: [B, 6, 19200]
    print(f"  Opt Values: {opt_v.shape}")   # Expected: [B, 200, 6]
    print(f"  Opt Errors: {opt_err.shape}") # Expected: [B, 200, 6]
    print(f"  Opt Masks:  {opt_mask.shape}")# Expected: [B, 200, 6]
    print(f"  Opt Times:  {opt_t.shape}")   # Expected: [B, 200]
    
    # Verify Unique GWs
    unique_gws = torch.unique(gw_indices)
    print(f"  Unique GWs in batch: {len(unique_gws)} (Should be Batch Size: {gw_s.shape[0]})")
    
    break

Building GW-to-Optical index mapping from /fred/oz016/bgao_kn/data/LSST_KN_BNS/combined_dataset.h5...
Mapping complete. Found 491 unique GW events.

Testing DataLoader...


Batch 0:
  GW Scalars: torch.Size([32, 9])
  GW Skymap:  torch.Size([32, 6, 19200])
  Opt Values: torch.Size([32, 200, 6])
  Opt Errors: torch.Size([32, 200, 6])
  Opt Masks:  torch.Size([32, 200, 6])
  Opt Times:  torch.Size([32, 200])
  Unique GWs in batch: 32 (Should be Batch Size: 32)
